# Enhanced World Model - Google Colab Training

Train world models for reinforcement learning on Google Colab with GPU acceleration.

**Architecture:** Vision (VQ-VAE/Identity) → Memory (LSTM/Transformer) → Controller (Actor-Critic)

**Supported Environments:**
- **Vector-based:** CartPole-v1, LunarLander-v3, Acrobot-v1
- **Image-based:** CarRacing-v3

**Quick Start:** Run all cells in order. Training starts automatically with sensible defaults.

In [ ]:
# @title 1. Setup Environment { display-mode: "form" }
# @markdown Check GPU and install dependencies

import subprocess
import sys
import os

# Check if running on Colab
IN_COLAB = "google.colab" in sys.modules

print("=" * 50)
print("Environment Setup")
print("=" * 50)

# Check GPU
!nvidia-smi -L 2>/dev/null || echo "No GPU detected - training will be slow!"

# Clone repo if not present
if not os.path.exists("Enhanced-World-Model"):
    print("\nCloning repository...")
    !git clone https://github.com/Larwive/Enhanced-World-Model.git
    %cd Enhanced-World-Model
else:
    %cd Enhanced-World-Model
    print("\nRepository already cloned, pulling latest...")
    !git pull

# Install dependencies
print("\nInstalling dependencies...")
!pip install -q gymnasium[box2d] torch torchvision tensorboard matplotlib numpy psutil swig

print("\n" + "=" * 50)
print("Setup complete!")
print("=" * 50)

In [ ]:
# @title 2. Initialize PyTorch { display-mode: "form" }
import sys

sys.path.insert(0, "src")

import torch
import numpy as np
import gymnasium as gym

# Device selection
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon (MPS)")
else:
    device = torch.device("cpu")
    print("Using CPU (training will be slower)")

print(f"\nPyTorch version: {torch.__version__}")
print(f"Device: {device}")

In [ ]:
# @title 3. Configuration { display-mode: "form" }
# @markdown ### Environment Settings
env_name = "CartPole-v1"  # @param ["CartPole-v1", "LunarLander-v3", "CarRacing-v3", "Acrobot-v1"]
num_envs = 16  # @param {type:"slider", min:1, max:64, step:1}

# @markdown ### Model Architecture
vision_type = "auto"  # @param ["auto", "VQ_VAE", "JEPA", "Identity"]
memory_type = "LSTMMemory"  # @param ["LSTMMemory", "TemporalTransformer"]
controller_type = "DeepDiscreteController"  # @param ["DiscreteModelPredictiveController", "DeepDiscreteController"]

# @markdown ### Training Settings
max_epochs = 500  # @param {type:"slider", min:100, max:5000, step:100}
learning_rate = 0.001  # @param {type:"number"}

# Auto-configuration based on environment
is_image_env = "CarRacing" in env_name or "Atari" in env_name

if vision_type == "auto":
    vision_type = "JEPA" if is_image_env else "Identity"

# Adjust num_envs for image-based envs (more memory intensive)
if is_image_env and num_envs > 8:
    num_envs = 8
    print(f"Note: Reduced num_envs to {num_envs} for image-based environment")

print("=" * 50)
print("Configuration")
print("=" * 50)
print(f"Environment:  {env_name}")
print(f"Vision:       {vision_type}")
print(f"Memory:       {memory_type}")
print(f"Controller:   {controller_type}")
print(f"Parallel envs: {num_envs}")
print(f"Max epochs:   {max_epochs}")
print(f"Learning rate: {learning_rate}")
print("=" * 50)

In [ ]:
# @title 4. Load Model Components { display-mode: "form" }
import vision
import memory
import controller
import reward_predictor
from utils.registry import discover_modules

# Discover available model classes
VISION_REGISTRY = discover_modules(vision)
MEMORY_REGISTRY = discover_modules(memory)
CONTROLLER_REGISTRY = discover_modules(controller)
REWARD_PREDICTOR_REGISTRY = discover_modules(reward_predictor)

print("Available models:")
print(f"  Vision:     {list(VISION_REGISTRY.keys())}")
print(f"  Memory:     {list(MEMORY_REGISTRY.keys())}")
print(f"  Controller: {list(CONTROLLER_REGISTRY.keys())}")
print(f"  Reward:     {list(REWARD_PREDICTOR_REGISTRY.keys())}")

In [ ]:
# @title 5. Create Environment { display-mode: "form" }
# Create vectorized environment
envs = gym.make_vec(env_name, num_envs=num_envs)

obs_space = envs.single_observation_space
action_space = envs.single_action_space
is_image_based = len(obs_space.shape) == 3
is_discrete = isinstance(action_space, gym.spaces.Discrete)

print("=" * 50)
print("Environment Info")
print("=" * 50)
print(f"Name:         {env_name}")
print(f"Observation:  {obs_space}")
print(f"Action:       {action_space}")
print(f"Image-based:  {is_image_based}")
print(f"Discrete:     {is_discrete}")
print("=" * 50)

In [ ]:
# @title 6. Configure Model Dimensions { display-mode: "form" }
# Vision configuration
if is_image_based:
    # Image: (H, W, C) -> (C, H, W) for PyTorch
    obs_shape = obs_space.shape
    input_shape = (obs_shape[2], obs_shape[0], obs_shape[1])
    embed_dim = 64
    vision_args = {
        "hidden_dim": 256,
        "output_dim": input_shape[0],  # For VQ_VAE
        "embed_dim": embed_dim,
        "predictor_dim": 256,  # For JEPA
    }
else:
    # Vector observation
    input_shape = obs_space.shape
    embed_dim = obs_space.shape[0]
    vision_args = {"embed_dim": embed_dim}

# Action dimension
action_dim = action_space.n if is_discrete else action_space.shape[0]

# Memory configuration
d_model = 128
memory_args = {
    "d_model": d_model,
    "latent_dim": embed_dim,
    "action_dim": action_dim,
    "nhead": 8,  # for Transformer
    "num_layers": 1,  # for LSTM
}

# Controller configuration
controller_args = {
    "action_dim": action_dim,
    "hidden_dim": 128,
    "num_layers": 2,
}

print("Model dimensions:")
print(f"  Input shape:    {input_shape}")
print(f"  Embed dim:      {embed_dim}")
print(f"  Memory d_model: {d_model}")
print(f"  Action dim:     {action_dim}")

In [ ]:
# @title 7. Create World Model { display-mode: "form" }
from WorldModel import WorldModel

# Create the world model
world_model = WorldModel(
    vision_model=VISION_REGISTRY[vision_type],
    memory_model=MEMORY_REGISTRY[memory_type],
    controller_model=CONTROLLER_REGISTRY[controller_type],
    input_shape=input_shape,
    vision_args=vision_args,
    memory_args=memory_args,
    controller_args=controller_args,
)

# Add reward predictor
world_model.set_reward_predictor(REWARD_PREDICTOR_REGISTRY["LinearPredictor"])

# Move entire model (including reward predictor) to device
world_model = world_model.to(device)

# Count parameters
total_params = sum(p.numel() for p in world_model.parameters())
trainable_params = sum(p.numel() for p in world_model.parameters() if p.requires_grad)

print("=" * 50)
print("World Model Created")
print("=" * 50)
print(f"Vision:      {vision_type}")
print(f"Memory:      {memory_type}")
print(f"Controller:  {controller_type}")
print(f"Parameters:  {total_params:,} total ({trainable_params:,} trainable)")
print(f"Device:      {device}")
print("=" * 50)

In [ ]:
# @title 8. Train! { display-mode: "form" }
from pathlib import Path
from train import train

# Create save directory
save_dir = Path("./saved_models/")
save_dir.mkdir(exist_ok=True)

print("=" * 50)
print("Starting Training")
print("=" * 50)
print(f"Environment:   {env_name}")
print(f"Parallel envs: {num_envs}")
print(f"Max epochs:    {max_epochs}")
print(f"Learning rate: {learning_rate}")
print(f"Device:        {device}")
print("=" * 50)
print()

# Train the model
train(
    model=world_model,
    envs=envs,
    max_iter=max_epochs,
    device=device,
    learning_rate=learning_rate,
    save_path=save_dir,
    use_tensorboard=True,
)

print("\nTraining complete!")

In [ ]:
# @title 9. Save Final Model { display-mode: "form" }
from datetime import datetime

# Save with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_name = save_dir / f"{env_name}_{timestamp}_final.pt"

world_model.save(save_name, obs_space, action_space)

print(f"Model saved to: {save_name}")
print(f"Training iterations: {world_model.iter_num}")
print(f"Episodes completed: {world_model.nb_experiments}")

In [ ]:
# @title 10. Evaluate Model { display-mode: "form" }
import matplotlib.pyplot as plt


def evaluate(model, env_name, num_episodes=5, device=device):
    """Evaluate trained model and show sample frames."""
    env = gym.make(env_name, render_mode="rgb_array")
    model.eval()

    is_img = len(env.observation_space.shape) == 3
    results = []

    for ep in range(num_episodes):
        obs, _ = env.reset()
        done = False
        total_reward = 0
        steps = 0
        frames = []

        # Reset model memory
        model.a_prev = None
        if hasattr(model.memory, "h_state"):
            model.memory.h_state = None
            model.memory.c_state = None

        while not done:
            # Prepare observation
            if is_img:
                obs_t = torch.from_numpy(obs).float().permute(2, 0, 1).unsqueeze(0) / 255.0
            else:
                obs_t = torch.from_numpy(obs).float().unsqueeze(0)
            obs_t = obs_t.to(device)

            # Get action
            with torch.no_grad():
                action = model(obs_t, env.action_space, is_img, return_losses=False)
                action = action.cpu().numpy()[0]

            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward
            steps += 1

            if steps % 10 == 0:
                frames.append(env.render())

        results.append({"reward": total_reward, "steps": steps, "frames": frames})
        print(f"Episode {ep + 1}: Reward = {total_reward:.1f}, Steps = {steps}")

    env.close()

    # Show sample frames from last episode
    if results[-1]["frames"]:
        frames = results[-1]["frames"]
        n_frames = min(5, len(frames))
        fig, axes = plt.subplots(1, n_frames, figsize=(3 * n_frames, 3))
        if n_frames == 1:
            axes = [axes]
        indices = np.linspace(0, len(frames) - 1, n_frames, dtype=int)
        for i, idx in enumerate(indices):
            axes[i].imshow(frames[idx])
            axes[i].axis("off")
        plt.suptitle(f"Last Episode - Reward: {results[-1]['reward']:.1f}")
        plt.tight_layout()
        plt.show()

    rewards = [r["reward"] for r in results]
    print(f"\nMean Reward: {np.mean(rewards):.1f} +/- {np.std(rewards):.1f}")
    return results


# Run evaluation
results = evaluate(world_model, env_name, num_episodes=5)

In [ ]:
# @title 11. List Saved Models { display-mode: "form" }
print("Saved models:")
print("-" * 50)
for f in sorted(save_dir.glob("*.pt")):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name} ({size_mb:.1f} MB)")

In [ ]:
# @title 12. Download Model (Colab) { display-mode: "form" }
# @markdown Downloads the most recent model to your computer

try:
    from google.colab import files

    model_files = sorted(save_dir.glob("*.pt"))
    if model_files:
        latest = model_files[-1]
        files.download(str(latest))
        print(f"Downloaded: {latest.name}")
    else:
        print("No models found")
except ImportError:
    print("Not running on Colab - copy files manually from ./saved_models/")

In [ ]:
# @title 13. Load Saved Model (Optional) { display-mode: "form" }
# @markdown Uncomment and set the path to load a previously saved model

# model_path = "./saved_models/CartPole-v1_20241225_123456.pt"
# world_model.load(model_path, obs_space, action_space)
# print(f"Loaded model from: {model_path}")
# print(f"  Iterations: {world_model.iter_num}")
# print(f"  Episodes:   {world_model.nb_experiments}")

---

## Quick Reference

### Recommended Configurations

| Environment | Vision | Memory | Controller | num_envs | Epochs |
|------------|--------|--------|------------|----------|--------|
| CartPole-v1 | Identity | LSTMMemory | DeepDiscreteController | 32 | 500 |
| LunarLander-v3 | Identity | LSTMMemory | DeepDiscreteController | 16 | 1000 |
| CarRacing-v3 | **JEPA** | LSTMMemory | ContinuousModelPredictiveController | 4 | 3000 |
| CarRacing-v3 | VQ_VAE | LSTMMemory | ContinuousModelPredictiveController | 4 | 3000 |
| Acrobot-v1 | Identity | LSTMMemory | DeepDiscreteController | 32 | 500 |

### Vision Models for Image Environments

| Model | Description | Best For |
|-------|-------------|----------|
| **JEPA** | Joint Embedding Predictive Architecture - learns in latent space, no reconstruction | Sample-efficient RL, task-relevant features |
| **VQ_VAE** | Vector Quantized VAE - discrete latent codes with reconstruction | When you need interpretable latents |

### Model Architecture

```
Observation → Vision (encode) → z_t → Memory (LSTM/Transformer) → h_t
                                 ↓            ↓
                           [z_t, h_t] → Controller → Action
                                 ↓
                         Reward Predictor → Predicted Reward
```

### Tips

1. **Start simple**: Use CartPole-v1 first to verify setup works
2. **JEPA vs VQ_VAE**: JEPA is more sample efficient (no reconstruction loss), try it first for image envs
3. **Monitor TensorBoard**: Watch for stable loss curves
4. **GPU memory**: Reduce `num_envs` if you run out of memory
5. **Save often**: Models auto-save on best loss, but manual saves help